# Pydantic

Pydantic is a popular Python library used for data validation and settings management

## WHy Data Validation is Necassary 
Data validation means checking that the data coming into our system is correct, safe, and in the expected format before we use it.

Catches Errors Early, Protects Your Database

In [1]:
%pip install -qU pydantic pydantic-settings

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pydantic import BaseModel

In [3]:
class User(BaseModel):
    id: int
    name: str
    is_active: bool = True   # default value -> field becomes optional
 

In [11]:
user = User(id="1", name="Zeeshan", is_active="true")
print(user)                 # id=1 name='Zeeshan' is_active=True
print(type(user.id)) 

user.name = "Zeeshan Ali"

id=1 name='Zeeshan' is_active=True
<class 'int'>


In [12]:
print(user)                 # id=1 name='Zeeshan Ali' is_active=True

user.name = 123
print(user)              

id=1 name='Zeeshan Ali' is_active=True
id=1 name=123 is_active=True


In [13]:
 
from pydantic import Field


class Product(BaseModel):
    name: str = Field(min_length=2, max_length=50)
    price: float = Field(gt=0, description="Price must be positive")
    quantity: int = Field(ge=0, default=0)
    sku: str = Field(alias="SKU")  # accept JSON key "SKU", store as .sku
 

In [15]:
 
p = Product(name="Laptop", price=999.99, SKU="LAP-001")
print(p)
print(p.sku)  
 

name='Laptop' price=999.99 quantity=0 sku='LAP-001'
LAP-001


In [16]:

from typing import Optional, Union


class Profile(BaseModel):
    bio: Optional[str] = None          # can be None, defaults to None
    contact: Union[str, int]           # can be a str OR an int
    website: str | None = None         # modern syntax (Python 3.10+), same as Optional
 

In [17]:
 
print(Profile(contact="zeeshan@example.com"))
print(Profile(contact=123456))
 

bio=None contact='zeeshan@example.com' website=None
bio=None contact=123456 website=None


In [27]:
class Customer(BaseModel):
    name: str
    tags: list[str] = []             # list of primitives
    orders: list[dict] = []       # list of dicts
 

In [28]:
customer = Customer(
    name="Zeeshan Ali",
    tags=["vip", "instructor"],
)
print(customer)
print(customer.tags)     # nested attribute access, default applied

name='Zeeshan Ali' tags=['vip', 'instructor'] orders=[]
['vip', 'instructor']


### Custom Validators

In [31]:
from pydantic import field_validator, model_validator

In [33]:
class SignupForm(BaseModel):
    username: str
    password: str
    confirm_password: str
    email: str
 
    # Runs on a SINGLE field, after Pydantic's own type validation
    @field_validator("username")
    @classmethod
    def username_no_spaces(cls, v: str) -> str:
        if " " in v:
            raise ValueError("username must not contain spaces")
        return v.lower()  # validators can also TRANSFORM the value
    
    # Runs on a SINGLE field, after Pydantic's own type validation
    @field_validator("password")
    @classmethod
    def password_basic_requirements(cls, v: str) -> str:
        if len(v) < 8:
            raise ValueError("password must be at least 8 characters long")
        if " " in v:
            raise ValueError("password must not contain spaces")
        return v
 
    # Runs on the WHOLE model, after all fields are individually validated
    @model_validator(mode="after")
    def passwords_match(self):
        if self.password != self.confirm_password:
            raise ValueError("password and confirm_password do not match")
        return self

In [36]:
form = SignupForm(
    username="Zeeshan Ali".replace(" ", ""),
    password="secret123",
    confirm_password="secret123",
    email="zeeshan@example.com",
)
print(form)

username='zeeshanali' password='secret123' confirm_password='secret123' email='zeeshan@example.com'


### Compute Field Validation

In [38]:
from pydantic import computed_field
from decimal import Decimal

In [40]:
class Order(BaseModel):
    unit_price: Decimal
    quantity: int
 
    @computed_field
    @property
    def total(self) -> Decimal:
        return self.unit_price * self.quantity
 


In [44]:
 
order = Order(unit_price=19.99, quantity=3)
print(order)

print(order.total)  # computed property

unit_price=Decimal('19.99') quantity=3 total=Decimal('59.97')
59.97


# Enums 

An enum (short for enumeration) is a specialized data type used to define a set of named, immutable constants.

In [ ]:
from enum import Enum

class Role(str, Enum):
    ADMIN = "admin"
    INSTRUCTOR = "instructor"
    STUDENT = "student"
 


In [21]:
class Account(BaseModel):
    username: str
    role: Role = Role.STUDENT
 
 
acc = Account(username="zeeshan", role="instructor")
print(acc)
print(acc.role is Role.INSTRUCTOR)  # True

username='zeeshan' role=<Role.INSTRUCTOR: 'instructor'>
True
